**INF201 - Deliverable 4**: Kachina Aschenberg Øwre & Seline Heggelund Knutsen

**GitHub repository:** https://github.com/kacoew/INF201-CAs (the code is in the folder `Deliverable_4`)

In [1]:
# SETUP

# re-import the .py files every time a cell runs, so saved changes are used
%load_ext autoreload
%autoreload 2

# sys is used to run pytest with the same python as this notebook
import sys

# Code shows a .py file with colours in the notebook (and in the pdf)
from IPython.display import Code

## Task 1: Temperatures

The `Temperature` class stores a temperature as a value and a unit: Celsius (`"C"`), Fahrenheit (`"F"`) or Kelvin (`"K"`). The unit is case-insensitive, and a `ValueError` is raised if the unit is invalid or the temperature is below absolute zero. The class can convert between all three units, and it can compare temperatures with different units, because all comparisons are done in Kelvin with a tolerance of `1e-9`.

**Design choices**
* The unit defaults to Celsius, because the task asks us to test a default unit without saying which one.
* `value` and `unit` are read-only: they are stored as `_value` and `_unit` and exposed with `@property` (no setter).
* `to_fahrenheit()` and `to_kelvin()` convert via Celsius, which gives the same result as the formulas in the task.
* Kelvin is printed without the ° sign (`300.00 K`), as in the SI system.

The code is in `temperature.py`, shown below. The sections are numbered 1–6 like the task text.

In [2]:
# Show the code in temperature.py (the file is the original - edit it there)
Code(filename="temperature.py", language="python")

"""
INF201 - Deliverable 4
Task 1: Temperature Class

A Python class for representing, converting, and comparing
temperatures in Celsius (C), Fahrenheit (F), and Kelvin (K).

Features:
    1. Initialization and validation:
       - Default unit: Celsius
       - Case-insensitive units, only C, F and K are accepted
       - Rejects temperatures below absolute zero (0 K)

    2. Tolerance:
       - Uses an absolute tolerance of 1e-9 for float comparisons

    3. Read-only value and unit attributes

    4. Temperature conversions:
       - to_celsius(), to_fahrenheit(), to_kelvin()
       - convert_to(target_unit)

    5. String representations:
       - __str__(): Human-readable temperature
       - __repr__(): Unambiguous object representation

    6. Comparison operators:
       - ==, !=, <, <=, >, >= across different units
"""


# IMPORTS

# lets us write "-> Temperature" inside the class before the class is finished
from __future__ import annotations


# CONSTANTS

# the only units the class accepts
VALID_UNITS = ("C", "F", "K")

# two temperatures closer than this count as equal (floats are never exact)
TOLERANCE = 1e-9


# TEMPERATURE CLASS

class Temperature:
    """A temperature with a value and unit."""


    # 1. INITIALIZATION AND VALIDATION

    # runs when a new Temperature is made, e.g. Temperature(25, "c")
    def __init__(self, value: float, unit: str = "C") -> None:
        """
        Initialise Temperature.

        Args:
            value (float): The temperature value.
            unit (str): The unit of the temperature ("C", "F", or "K"). Default is "C".

        Raises:
            ValueError: If the unit is not valid or if the temperature is below absolute zero.
        """

        # save unit and value as private attributes (_ = not meant to be changed)
        # float, so 25 is shown as 25.0
        self._unit = self._check_unit(unit)
        self._value = float(value)

        # temp must not be below absolute zero, checked in kelvin so it works for all units
        if self.to_kelvin() < -TOLERANCE:
            raise ValueError("Temperature below absolute zero.")


    # helper used by both __init__ and convert_to, so the unit check is only written once
    def _check_unit(self, unit: str) -> str:
        """Check that the unit is valid and return it in upper case."""

        # unit must be a string, else .upper() crashes
        if not isinstance(unit, str) or unit.upper() not in VALID_UNITS:
            raise ValueError(f"Invalid unit {unit!r}. Valid units are {VALID_UNITS}.")

        # make unit case-insensitive
        return unit.upper()


    # 2. TOLERANCE

    # true if the difference between a and b is at most 1e-9
    def _is_close(self, a: float, b: float) -> bool:
        """Check if two values are equal within the tolerance."""
        return abs(a - b) <= TOLERANCE


    # 3. READ-ONLY PROPERTIES

    # @property lets us read temp.value and temp.unit,
    # but there is no setter, so temp.value = 5 gives an AttributeError

    @property
    def value(self) -> float:
        """Return the temperature value."""
        return self._value

    @property
    def unit(self) -> str:
        """Return the temperature unit."""
        return self._unit


    # 4. TEMPERATURE CONVERSIONS

    # one formula for each unit
    def to_celsius(self) -> float:
        """Convert the temperature to Celsius."""
        if self.unit == "F":
            return (self.value - 32) * 5 / 9
        if self.unit == "K":
            return self.value - 273.15
        return self.value  # already celsius

    # if already fahrenheit: return as it is, else go via celsius
    def to_fahrenheit(self) -> float:
        """Convert the temperature to Fahrenheit."""
        if self.unit == "F":
            return self.value  # already fahrenheit
        return self.to_celsius() * 9 / 5 + 32  # via celsius

    # if already kelvin: return as it is, else go via celsius
    def to_kelvin(self) -> float:
        """Conver

### Demonstration of the Temperature class

In [3]:
from temperature import Temperature

# 1. initialization and validation
body = Temperature(37, "c")  # lowercase unit is accepted
print("Default unit:", Temperature(25).unit)
print("Unit 'c' is saved as:", body.unit)

for value, unit in [(-300, "C"), (25, "X")]:  # below absolute zero, invalid unit
    try:
        Temperature(value, unit)
    except ValueError as error:
        print(f"Temperature({value}, {unit!r}) -> ValueError: {error}")

# 2. tolerance
print("25 °C == 25.0000000001 °C:", Temperature(25) == Temperature(25 + 1e-10))

# 3. read-only
try:
    body.value = 40
except AttributeError as error:
    print("body.value = 40 -> AttributeError:", error)

# 4. conversions
print(f"37 °C in F: {body.to_fahrenheit():.2f} | in K: {body.to_kelvin():.2f}")
print("convert_to('f'):", body.convert_to("f"))

# 5. string representations
print("str: ", str(body))
print("repr:", repr(body))

# 6. comparisons across units
print("0 °C == 32 °F:", Temperature(0, "C") == Temperature(32, "F"))
print("0 °C < 300 K:", Temperature(0, "C") < Temperature(300, "K"))
print("100 °C >= 212 °F:", Temperature(100, "C") >= Temperature(212, "F"))

Default unit: C
Unit 'c' is saved as: C
Temperature(-300, 'C') -> ValueError: Temperature below absolute zero.
Temperature(25, 'X') -> ValueError: Invalid unit 'X'. Valid units are ('C', 'F', 'K').
25 °C == 25.0000000001 °C: True
body.value = 40 -> AttributeError: property 'value' of 'Temperature' object has no setter
37 °C in F: 98.60 | in K: 310.15
convert_to('f'): 98.60 °F
str:  37.00 °C
repr: Temperature(37.0, 'C')
0 °C == 32 °F: True
0 °C < 300 K: True
100 °C >= 212 °F: True


### Tests for Task 1 (`test_temperature.py`)

The tests are grouped like the six test points in the task. They are run with pytest below.

In [4]:
# Show the code in test_temperature.py (the file is the original - edit it there)
Code(filename="test_temperature.py", language="python")

"""
INF201 - Deliverable 4 - test_temperature.py
Task 1: Temperature Class Tests

Tests:
    1. Initialization: Default unit, case-insensitive units, and read-only behavior.
    2. Validation: Invalid unit strings and temperatures below absolute zero in "C", "F", and "K".
    3. Conversions: Test conversion accuracy across the five reference points.
    4. Round-trip conversions: Converting "C" -> "F" -> "K" -> "C" reproduces the original Temperature.
    5. Cross-unit comparisons: Equality and ordering (<, <=, >, >=) across mixed units.
    6. Formatting: Outputs of `str()` and `repr()`

Run with:
    python -m pytest test_temperature.py -v
"""


# IMPORTS

# pytest runs the tests, Temperature is the class we test
import pytest

from temperature import Temperature


# CONSTANTS

# reference points from the task: (celsius, fahrenheit, kelvin)
# used by the conversion test, so the numbers are only written once
REFERENCE_POINTS = [
    (-273.15, -459.67, 0.0),   # absolute zero
    (-40.0, -40.0, 233.15),    # celsius-fahrenheit crossover
    (0.0, 32.0, 273.15),       # freezing point of water
    (37.0, 98.6, 310.15),      # human body temperature
    (100.0, 212.0, 373.15),    # boiling point of water
]


# 1. INITIALIZATION

# no unit given -> celsius
def test_default_unit():
    """Test that Celsius is the default unit."""

    assert Temperature(25).unit == "C"


# parametrize runs the same test once for each unit in the list
# small letters should be saved as capital letters
@pytest.mark.parametrize("unit", ["c", "f", "k", "C", "F", "K"])
def test_case_insensitivity(unit):
    """Test that temperature units are case-insensitive."""

    assert Temperature(300, unit).unit == unit.upper()


# pytest.raises checks that the code inside gives the expected error
# changing value or unit should give an AttributeError
def test_read_only_behavior():
    """Test that value and unit cannot be modified."""

    temp = Temperature(25, "C")

    with pytest.raises(AttributeError):
        temp.value = 30

    with pytest.raises(AttributeError):
        temp.unit = "F"


# 2. VALIDATION

# wrong text and wrong types (5, None) should both give ValueError
@pytest.mark.parametrize("unit", ["X", "Celsius", "", 5, None])
def test_invalid_unit(unit):
    """Test that invalid units raise ValueError."""

    with pytest.raises(ValueError):
        Temperature(25, unit)


# one value just below absolute zero in each unit
@pytest.mark.parametrize("value, unit", [(-273.16, "C"), (-459.68, "F"), (-0.01, "K")])
def test_below_absolute_zero(value, unit):
    """Test that temperatures below absolute zero are rejected."""

    # just below the limit, so a small bug is caught
    with pytest.raises(ValueError):
        Temperature(value, unit)


# absolute zero itself is the limit, so it must be allowed
def test_absolute_zero_boundaries():
    """Test that absolute zero itself is valid."""

    # no error should be raised
    Temperature(-273.15, "C")
    Temperature(-459.67, "F")
    Temperature(0, "K")


# convert_to uses the same unit check as __init__
def test_invalid_target_unit():
    """Test that convert_to rejects invalid target units."""

    with pytest.raises(ValueError):
        Temperature(25).convert_to("X")


# 3. TEMPERATURE CONVERSIONS

# runs once for each of the five reference points
# pytest.approx allows tiny rounding errors in floats
@pytest.mark.parametrize("celsius, fahrenheit, kelvin", REFERENCE_POINTS)
def test_conversion_accuracy(celsius, fahrenheit, kelvin):
    """Test conversions at five reference temperatures."""

    # start in each unit, and check all three conversions
    for temp in [Temperature(celsius, "C"), Temperature(fahrenheit, "F"), Temperature(kelvin, "K")]:
        assert temp.to_celsius() == pytest.approx(celsius)
        assert temp.to_fahrenheit() == pytest.approx(fahrenheit)
        assert temp.to_kelvin() == pytest.approx(kelvin)


# convert_to should give a new object with the right unit and value
def te

In [5]:
# Run the Task 1 tests (uses the same Python as this notebook)
!{sys.executable} -m pytest -v test_temperature.py

============================= test session starts ==============================
platform darwin -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0 -- /Users/kachinaaschenberg/Documents/NMBU/INF201/INF201/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kachinaaschenberg/Documents/NMBU/INF201/INF201/Deliverable_4
plugins: platformdirs-4.13.0
collected 35 items                                                             

test_temperature.py::test_default_unit PASSED                            [  2%]
test_temperature.py::test_case_insensitivity[c] PASSED                   [  5%]
test_temperature.py::test_case_insensitivity[f] PASSED                   [  8%]
test_temperature.py::test_case_insensitivity[k] PASSED                   [ 11%]
test_temperature.py::test_case_insensitivity[C] PASSED                   [ 14%]
test_temperature.py::test_case_insensitivity[F] PASSED                   [ 17%]
test_temperature.py::test_case_insensitivity[K] PASSED                   [ 20%]
test_temperat

## Task 2: Fractions


In [6]:
"""Fraction class with comparisons and arithmetic special methods (Task 2)."""

from __future__ import annotations
import math

class Fraction:
    """A fraction with a numerator and denominator"""

    def __init__(self, numerator: int, denominator: int) -> None:
        """
        Initialise the Fraction.

        Args:
            numerator (int): The numerator
            denominator (int): The denominator

        Raises:
            ValueError: If the denominator is zero.
        """
        if denominator <= 0:
            raise ValueError("Denominator cannot be zero")
        self.numerator = numerator
        self.denominator = denominator

        if not isinstance(numerator, int):
            raise TypeError('Numerator must be an integer')

        if not isinstance(denominator, int):
            raise TypeError('Denominator must be an interger')

    def __str__(self) -> str:
        '''
        Return a human-readable string representation of the fraction

        Returns:
            str: The fraction in the format numerator/denominator
        '''
        return f'{self.numerator}/{self.denominator}'

    def __repr__(self) -> str:
        """
        Return an official string representation of the fraction

        Returns:
            str: A string representation of the Fraction object
        """
        return f"Fraction({self.numerator}, {self.denominator})"

    def __eq__(self, other: 'Fraction') -> bool:
        '''
        Check if two fractions are equal

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if the fractions are equal, otherwise False
        '''
        return self.numerator * other.denominator == other.numerator * self.denominator

    def __ne__(self, other: "Fraction") -> bool:
        """
        Check if two fractions are not equal

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if the fractions are not equal, otherwise False
        """
        return not self.__eq__(other)

    def __lt__(self, other: "Fraction") -> bool:
        """
        Check if this fraction is less than another fraction

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if this fraction is less than the other fraction
        """
        return self.numerator * other.denominator < other.numerator * self.denominator

    def __le__(self, other: "Fraction") -> bool:
        """
        Check if this fraction is less than or equal to another fraction

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if this fraction is less than or equal to the other fraction
        """
        return self < other or self == other

    def __gt__(self, other: "Fraction") -> bool:
        """
        Check if this fraction is greater than another fraction

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if this fraction is greater than the other fraction
        """
        return self.numerator * other.denominator > other.numerator * self.denominator

    def __ge__(self, other: "Fraction") -> bool:
        """
        Check if this fraction is greater than or equal to another fraction

        Args:
            other (Fraction): The fraction to compare with

        Returns:
            bool: True if this fraction is greater than or equal to the other fraction
        """
        return not self < other

    def __add__(self, other: "Fraction") -> "Fraction":
        """
        Add two fractions

        Args:
            other (Fraction): The fraction to add

        Returns:
            Fraction: The result of the addition
        """
        numerator = (
            self.numerator * other.denominator
            + other.numerator * self.denominator
        )

        denominator = self.denominator * other.denominator

        return Fraction(numerator, denominator)

    def __sub__(self, other: "Fraction") -> "Fraction":
        """
        Subtract one fraction from another

        Args:
            other (Fraction): The fraction to subtract

        Returns:
            Fraction: The result of the subtraction
        """
        numerator = (
            self.numerator * other.denominator
            - other.numerator * self.denominator
        )

        denominator = self.denominator * other.denominator

        return Fraction(numerator, denominator)

    def __mul__(self, other: "Fraction") -> "Fraction":
        """
        Multiply two fractions

        Args:
            other (Fraction): The fraction to multiply by

        Returns:
            Fraction: The result of the multiplication
        """
        return Fraction(
            self.numerator * other.numerator,
            self.denominator * other.denominator
        )

    def __truediv__(self, other: "Fraction") -> "Fraction":
        """
        Divide one fraction by another

        Args:
            other (Fraction): The divisor fraction

        Returns:
            Fraction: The result of the division

        Raises:
            ValueError: If attempting to divide by zero
        """
        if other.numerator == 0:
            raise ValueError("Cannot divide by zero")

        return Fraction(
            self.numerator * other.denominator,
            self.denominator * other.numerator
        )

### Demonstration of the Fraction class

In [7]:
from fraction import Fraction

f1 = Fraction(1, 2)
f2 = Fraction(3, 4)

print(f1)
print(f2)

print(f1 + f2)
print(f1 - f2)
print(f1 * f2)
print(f1 / f2)

print(f1 == f2)
print(f1 != f2)
print(f1 < f2)
print(f1 > f2)

1/2
3/4
10/8
-2/8
3/8
4/6
False
True
True
False


### Tests for Task 2 (`test_fraction.py`)

In [8]:
"""Tests for the Fraction class."""

import pytest
from fraction import Fraction


def test_fraction_creation():
    """Test creating a valid fraction."""
    f = Fraction(3, 4)

    assert f.numerator == 3
    assert f.denominator == 4


def test_zero_denominator():
    """Test that a denominator of zero raises ValueError."""
    with pytest.raises(ValueError):
        Fraction(1, 0)


def test_negative_denominator():
    """Test that a negative denominator raises ValueError."""
    with pytest.raises(ValueError):
        Fraction(1, -2)


def test_non_integer_numerator():
    """Test that a non-integer numerator raises TypeError."""
    with pytest.raises(TypeError):
        Fraction(1.5, 2)


def test_non_integer_denominator():
    """Test that a non-integer denominator raises TypeError."""
    with pytest.raises(TypeError):
        Fraction(1, 2.5)


def test_string_representation():
    """Test the __str__ method."""
    assert str(Fraction(3, 4)) == "3/4"


def test_repr_representation():
    """Test the __repr__ method."""
    assert repr(Fraction(3, 4)) == "Fraction(3, 4)"


def test_equal():
    """Test equality comparison."""
    assert Fraction(1, 2) == Fraction(2, 4)


def test_not_equal():
    """Test inequality comparison."""
    assert Fraction(1, 2) != Fraction(3, 4)


def test_less_than():
    """Test less than comparison."""
    assert Fraction(1, 3) < Fraction(1, 2)


def test_less_than_or_equal():
    """Test less than or equal comparison."""
    assert Fraction(1, 2) <= Fraction(2, 4)


def test_greater_than():
    """Test greater than comparison."""
    assert Fraction(3, 4) > Fraction(1, 2)


def test_greater_than_or_equal():
    """Test greater than or equal comparison."""
    assert Fraction(3, 4) >= Fraction(3, 4)


def test_addition():
    """Test fraction addition."""
    result = Fraction(1, 2) + Fraction(1, 3)

    assert result == Fraction(5, 6)


def test_subtraction():
    """Test fraction subtraction."""
    result = Fraction(3, 4) - Fraction(1, 4)

    assert result == Fraction(8, 16)


def test_multiplication():
    """Test fraction multiplication."""
    result = Fraction(2, 3) * Fraction(3, 4)

    assert result == Fraction(6, 12)


def test_division():
    """Test fraction division."""
    result = Fraction(2, 3) / Fraction(4, 5)

    assert result == Fraction(10, 12)


def test_division_by_zero_fraction():
    """Test division by a zero fraction."""
    with pytest.raises(ValueError):
        Fraction(1, 2) / Fraction(0, 1)

In [9]:
# Run the Task 2 tests
!pytest test_fraction.py

============================= test session starts ==============================
platform darwin -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/kachinaaschenberg/Documents/NMBU/INF201/INF201/Deliverable_4
plugins: platformdirs-4.13.0
collected 18 items                                                             

test_fraction.py ..................                                      [100%]

============================== 18 passed in 0.07s ==============================


## Summary of Gen AI use

<!-- TODO: Each of you writes a short, honest paragraph: which tool, what you asked about,
     what the AI did, and what you did yourselves. -->

**Kachina:** TODO

**Seline:** I used Microsoft Copilot as a guidance tool while completing this assignment. It helped me understand the task requirements, explained object\-oriented programming concepts, guided me through implementing the Fraction class and its special methods, and showed me how to create pytest tests.

The AI acted as a tutor throughout the process, while I was responsible for implementing, testing, and finalizing the code myself.

